# Warden semantic trigger filter

This notebook embeds a user's natural-language avoidance preference and a structured post. It compares them with cosine similarity instead of keyword matching.

In [4]:
# Run once if needed:
# %pip install sentence-transformers

import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

from pathlib import Path
import json
import torch
from sentence_transformers import SentenceTransformer, util

MODEL_PATH = Path('../models/semantic-filter')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model not found: {MODEL_PATH.resolve()}')

embedder = SentenceTransformer(str(MODEL_PATH), device=DEVICE)
print('device:', DEVICE)
print('embedding dimension:', embedder.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

device: cuda
embedding dimension: 384


C:\Users\Abhid\AppData\Local\Temp\ipykernel_25764\940735250.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print('embedding dimension:', embedder.get_sentence_embedding_dimension())


In [12]:
def post_to_text(post: dict) -> str:
    fields = [
        ('Title', post.get('title', '')),
        ('Caption', post.get('caption', '')),
        ('Alt text', post.get('alt_text', '')),
        ('Account', post.get('account_name', '')),
    ]
    return '\n'.join(f'{name}: {value}' for name, value in fields if value and str(value).strip())

def semantic_score(preference: str, post: dict, threshold: float = 0.25) -> dict:
    post_text = post_to_text(post)
    if not preference.strip() or not post_text.strip():
        raise ValueError('Both preference and post text must be non-empty')

    embeddings = embedder.encode(
        [preference, post_text], convert_to_tensor=True, normalize_embeddings=True
    )
    similarity = float(util.cos_sim(embeddings[0], embeddings[1]))
    return {
        'preference': preference,
        'post_text': post_text,
        'similarity': round(similarity, 4),
        'action': 'hide' if similarity >= threshold else 'show',
    }

In [ ]:
preference = 'i want to hide posts about exams and studying'
post = {
    "title": "How I prepared for my biggest academic challenge",
    "caption": "I made a revision schedule, practiced mock papers, and reviewed everything the night before.",
    "account_name": "Campus Stories",
    "alt_text": "Notebook and textbooks on a desk",
}

print(json.dumps(semantic_score(preference, post), indent=2))

{
  "preference": "i want to hide posts about exams and studying",
  "post_text": "Title: How I prepared for my biggest academic challenge\nCaption: I made a revision schedule, practiced mock papers, and reviewed everything the night before.\nAlt text: Notebook and textbooks on a desk\nAccount: Campus Stories",
  "similarity": 0.2919,
  "action": "hide"
}


The 0.35 threshold is only a demo placeholder, not a measured result. Calibrate it using posts that the user marks as relevant and irrelevant. Raw images still need OCR, image captioning, or a vision-text model before they can contribute semantic text similarity.